In [16]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from pathlib import Path

from sentinel.ml_logic.metrics import corrected_event_f05

from tqdm import tqdm

import itertools


# --- config
VAL_FRAC = 0.2  # temporal split
Q_LOW_GRID  = [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]
Q_HIGH_GRID = [0.99, 0.995, 0.996, 0.997, 0.998]

DATA_DIR        = "../data"
RAW_DIR         = DATA_DIR + "/raw"
train_path = RAW_DIR + "/train.parquet"
test_path = RAW_DIR + "/test.parquet"
column_names = ['channel_' + str(n) for n in range(41, 46+1)]


# --- 1. load only required columns
cols = ['id', 'is_anomaly'] + column_names

train = pq.read_table(train_path, columns=cols).to_pandas().set_index('id')
test  = pq.read_table(test_path,  columns=['id'] + column_names).to_pandas().set_index('id')

# --- 2. temporal split (no shuffle)
split_idx = int(len(train) * (1 - VAL_FRAC))

train_tr = train.iloc[:split_idx]
train_val = train.iloc[split_idx:]

X_tr = train_tr[column_names]
X_val = train_val[column_names]
y_val = train_val['is_anomaly'].values

# --- 3. threshold function
def predict_with_thresholds(df, low, high):
    return ((df < low) | (df > high)).any(axis=1).astype(int).values


In [17]:
# --- 4. grid search
best_score = -1
best_params = None
best_thresholds = None
best_metrics = None

grid = list(itertools.product(Q_LOW_GRID, Q_HIGH_GRID))

for q_low, q_high in tqdm(grid, desc="Threshold search"):
        
    low  = X_tr.quantile(q_low)
    high = X_tr.quantile(q_high)

    y_pred_val = predict_with_thresholds(X_val, low, high)

    metric = corrected_event_f05(y_val, y_pred_val)
    metric["f1"] = corrected_event_f05(y_val, y_pred_val, beta=1.0)["f_score"]
    score = metric["f_score"]

    if score > best_score:
        best_score = score
        best_params = (q_low, q_high)
        best_thresholds = (low, high)
        best_metrics = metric

print(f"Best params: q_low={best_params[0]}, q_high={best_params[1]}")
print(f"Best F0.5: {best_score:.4f}")

# --- 5. final model (fit on full train)
low, high = best_thresholds
print(best_metrics)

# --- 6. inference on test
y_pred_test = predict_with_thresholds(test[column_names], low, high)

Threshold search: 100%|██████████| 25/25 [07:19<00:00, 17.60s/it]

Best params: q_low=0.0001, q_high=0.997
Best F0.5: 0.6977
{'f_score': 0.697674, 'precision': 1.0, 'recall': 0.315789, 'tp_events': 12, 'fn_events': 26, 'fp_pred_events': 0, 'fp_samples': 0, 'tnr': 1.0, 'f1': 0.48}
